# Förberedelser

## Importera

In [57]:
import SPARQLWrapper as sparql
import pandas as pd

In [58]:
endpoint = "https://libris.kb.se/sparql"

# Analys

## Digitiseringar
Den digitiserade versionen av en fysisk resurs. 

I dagsläget finns en hänvisning, som lokal entitet, till det fysiska originalet i otherPhysicalFormat > describedBy > controlNumber .

Målet är att byta ut den lokala entiteten mot en länakd entitet i reproductionOf.

In [59]:
query = '''

PREFIX marc: <https://id.kb.se/marc/>
PREFIX : <https://id.kb.se/vocab/>

SELECT  ?instance ?recordControlNumber ?localControlNumber ?marcDisplayText WHERE {

	?record :mainEntity ?instance ;
    :controlNumber ?recordControlNumber .
    
  ?instance a :DigitalResource ;
    :otherPhysicalFormat ?localEntity .
    # :sameAs ?recordControlNumber .

  ?localEntity :describedBy [ a :Record;
     :controlNumber ?localControlNumber ] ;
     marc:displayText ?marcDisplayText .

  FILTER(REGEX(?localControlNumber, "^[0-9a-z]+$"))

  FILTER isBLANK(?localEntity) .

  FILTER(regex(?marcDisplayText, "(originalversion|orginalversion)", "i"))

}

'''

digitizations = sparql.get_sparql_dataframe(endpoint, query)
digitizations.info()
digitizations.head(1)

<class 'pandas.DataFrame'>
RangeIndex: 46544 entries, 0 to 46543
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   instance             46544 non-null  str  
 1   recordControlNumber  46544 non-null  str  
 2   localControlNumber   46544 non-null  str  
 3   marcDisplayText      46544 non-null  str  
dtypes: str(4)
memory usage: 1.4 MB


,instance,recordControlNumber,localControlNumber,marcDisplayText
0,https://libris.kb.se/hwg7pxp0fm8zbjcq#it,hwg7pxp0fm8zbjcq,3299611,Originalversion:


In [60]:
digitizations[digitizations["instance"] == "https://libris.kb.se/j2vbjjtv3r846hn#it"]

,instance,recordControlNumber,localControlNumber,marcDisplayText
33291,https://libris.kb.se/j2vbjjtv3r846hn#it,22698896,3121391,Originalversion


In [61]:
# Spara eventuella dubbletter för vidare analys. Spara i https://kbse.atlassian.net/browse/LXL-4790
# digitizations[digitizations.duplicated(keep=False)].to_csv("digitala_med_osynliga_blanknoder.csv")

# Preppa/städa
#digitizations.drop_duplicates(inplace=True)

#digitizations.info()
#digitizations.head(1)

### Finns det digitiseringar som hänviar till flera original? 🤯

In [62]:
repros_with_multiple_originals = digitizations[digitizations.duplicated(subset=["instance"], keep=False)]
repros_with_multiple_originals.value_counts("instance")

instance
https://libris.kb.se/5phq41ch47w3cxl#it    2
https://libris.kb.se/h1t2gj0t4jknmgg#it    2
https://libris.kb.se/h1t2wj4t55ctmvp#it    2
https://libris.kb.se/6qjz3wtj5j7lnc3#it    2
https://libris.kb.se/0jbqj6hb27c7kx3#it    2
Name: count, dtype: int64

Droppa dessa, för att istället hantera manuellt.

In [63]:
digitizations.drop_duplicates(subset="instance", keep=False, inplace=True)
digitizations.info()

<class 'pandas.DataFrame'>
Index: 46534 entries, 0 to 46543
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   instance             46534 non-null  str  
 1   recordControlNumber  46534 non-null  str  
 2   localControlNumber   46534 non-null  str  
 3   marcDisplayText      46534 non-null  str  
dtypes: str(4)
memory usage: 1.8 MB


In [64]:
#repros_with_multiple_originals.to_csv("reproduktioner_med_flera_original.tsv", sep="\t")

## Original
Den fysiska resurs som digitiserats.

In [65]:
query = '''

PREFIX marc: <https://id.kb.se/marc/>
PREFIX : <https://id.kb.se/vocab/>

SELECT ?instance ?recordControlNumber ?localControlNumber ?marcDisplayText WHERE {

	?record :mainEntity ?instance ;
    :controlNumber ?recordControlNumber .

  ?instance a :PhysicalResource ;
    :otherPhysicalFormat ?localEntity .
    # :sameAs ?recordControlNumber .

  FILTER isBLANK(?localEntity) .

  ?localEntity :describedBy [ a :Record;
     :controlNumber ?localControlNumber ] ;
     marc:displayText ?marcDisplayText .

  FILTER(REGEX(?localControlNumber, "^[0-9a-z]+$"))
  FILTER(regex(?marcDisplayText, "(^digit.*version)", "i"))
  FILTER(!regex(?note, "(del|1969)", "i"))

}

'''

originals = sparql.get_sparql_dataframe(endpoint, query)
originals.info()
originals.head(1)

<class 'pandas.DataFrame'>
RangeIndex: 44956 entries, 0 to 44955
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   instance             44956 non-null  str  
 1   recordControlNumber  44956 non-null  str  
 2   localControlNumber   44956 non-null  str  
 3   marcDisplayText      44956 non-null  str  
dtypes: str(4)
memory usage: 1.4 MB


,instance,recordControlNumber,localControlNumber,marcDisplayText
0,https://libris.kb.se/0h9w37wb2djzvrr#it,915820,htzk4vlsfwkz1mbz,Digitaliserad version:


In [66]:
# Spara eventuella dubbletter för vidare analys. Spara i https://kbse.atlassian.net/browse/LXL-4790
#originals[originals.duplicated(keep=False)].to_csv("fysiska_med_osynliga_blanknoder.csv")

# Preppa/städa
#originals.drop_duplicates(inplace=True)

#originals.info()
#originals.head(1)

### Finns det original som pekar mot flera digitiseringar?



In [67]:
originals_with_multiple_repros = originals[originals.duplicated(subset=["instance"], keep=False)]
originals_with_multiple_repros.info()
originals_with_multiple_repros.value_counts("instance")

<class 'pandas.DataFrame'>
Index: 631 entries, 103 to 44759
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   instance             631 non-null    str  
 1   recordControlNumber  631 non-null    str  
 2   localControlNumber   631 non-null    str  
 3   marcDisplayText      631 non-null    str  
dtypes: str(4)
memory usage: 24.6 KB


instance
https://libris.kb.se/xf742w281ml66nx#it     5
https://libris.kb.se/9sl8ljsm4d7rfjx#it     4
https://libris.kb.se/3ld3tvrf3dzzh56#it     4
https://libris.kb.se/zg8530895flm4dt#it     3
https://libris.kb.se/cvn7c43p40fm4b7#it     3
                                           ..
https://libris.kb.se/h0sf34st1mm7xql#it     2
https://libris.kb.se/m4xj7g1z0tczvpf#it     2
https://libris.kb.se/cv0p02vn9198v97h#it    2
https://libris.kb.se/l37x9rx9jc9kg7rm#it    2
https://libris.kb.se/m4xltfhz1mr8sf9#it     2
Name: count, Length: 310, dtype: int64

## Jämförelse

In [68]:
mutual_matches = digitizations.merge(
    originals,
    how="inner",
    left_on=["localControlNumber", "recordControlNumber"],
    right_on=["recordControlNumber", "localControlNumber"],
    suffixes=("_digital", "_physical")
)

# Helt identiska rader kan droppas
mutual_matches.drop_duplicates(inplace=True)

mutual_matches.info()
mutual_matches.head(5)

mutual_matches_to_print = mutual_matches.copy()
mutual_matches_to_print["instance_digital"] = mutual_matches_to_print["instance_digital"].str.extract(r"https://libris\.kb\.se/([a-zA-Z0-9]+)#it", expand=False)
mutual_matches_to_print["instance_physical"] = mutual_matches_to_print["instance_physical"].str.extract(r"https://libris\.kb\.se/([a-zA-Z0-9]+)#it", expand=False)
mutual_matches_to_print.to_csv("mutual_matches_ids.tsv", sep="\t", index=False)

<class 'pandas.DataFrame'>
RangeIndex: 43354 entries, 0 to 43353
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   instance_digital              43354 non-null  str  
 1   recordControlNumber_digital   43354 non-null  str  
 2   localControlNumber_digital    43354 non-null  str  
 3   marcDisplayText_digital       43354 non-null  str  
 4   instance_physical             43354 non-null  str  
 5   recordControlNumber_physical  43354 non-null  str  
 6   localControlNumber_physical   43354 non-null  str  
 7   marcDisplayText_physical      43354 non-null  str  
dtypes: str(8)
memory usage: 2.6 MB


In [69]:
# Vilka av de ömsesidigt matchande originalen har flera reproduktioner?
mutual_matches[mutual_matches.duplicated(subset=["instance_physical"], keep=False)].sort_values("instance_physical")

,instance_digital,recordControlNumber_digital,localControlNumber_digital,marcDisplayText_digital,instance_physical,recordControlNumber_physical,localControlNumber_physical,marcDisplayText_physical
6836,https://libris.kb.se/q779drh3nqx529m7#it,q779drh3nqx529m7,8073130,Originalversion:,https://libris.kb.se/0h96zbgb5vdzqxh#it,8073130,q779drh3nqx529m7,Digitaliserad version:
40223,https://libris.kb.se/m5zb7jvz100sc3z#it,19942139,8073130,Orginalversion:,https://libris.kb.se/0h96zbgb5vdzqxh#it,8073130,19942139,Digitaliserad version:
29114,https://libris.kb.se/vd69hjr61wwk7s9#it,13482516,10254730,Originalversion,https://libris.kb.se/0h99m4gb42hvxzm#it,10254730,13482516,Digitaliserad version
12973,https://libris.kb.se/l4x5w5lx3kmhxfp#it,17259928,10254730,Orginalversion,https://libris.kb.se/0h99m4gb42hvxzm#it,10254730,17259928,Digitaliserad version
22480,https://libris.kb.se/fxx7kklqc56rkwqd#it,fxx7kklqc56rkwqd,305440,Originalversion:,https://libris.kb.se/0h9wcmpb3ws6ffm#it,305440,fxx7kklqc56rkwqd,Digitaliserad version:
...,...,...,...,...,...,...,...,...
39839,https://libris.kb.se/cwp56tzp4h8zkn9#it,22548861,1596939,Originalversion:,https://libris.kb.se/zg8wxfn94jp0wjd#it,1596939,22548861,Digitaliserad version:
10438,https://libris.kb.se/vd6736v61wcjnfm#it,12339606,1598259,Originalversion:,https://libris.kb.se/zg8wxg492ws998j#it,1598259,12339606,Digitaliserad version:
32446,https://libris.kb.se/1kcnh4zc3qghn92#it,18219251,1598259,Originalversion,https://libris.kb.se/zg8wxg492ws998j#it,1598259,18219251,Digitaliserad version
33190,https://libris.kb.se/h1t77bjt5mtv2g3#it,20858155,1601799,Originalversion:,https://libris.kb.se/zg8wxl290sqvgts#it,1601799,20858155,Digitaliserad version:


In [70]:
# Vilka reproduktioner har flera original?
mutual_matches[mutual_matches.duplicated(subset=["instance_digital"], keep=False)]

,instance_digital,recordControlNumber_digital,localControlNumber_digital,marcDisplayText_digital,instance_physical,recordControlNumber_physical,localControlNumber_physical,marcDisplayText_physical


## Ensidiga matcher

In [71]:
digi_to_physical_matches = digitizations.merge(
    originals,
    how="inner",
    left_on=["localControlNumber"],
    right_on=["recordControlNumber"],
    suffixes=("_digital", "_physical")
)

digi_to_physical_matches.drop_duplicates(inplace=True)

digi_to_physical_matches.info()

<class 'pandas.DataFrame'>
RangeIndex: 44828 entries, 0 to 44827
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   instance_digital              44828 non-null  str  
 1   recordControlNumber_digital   44828 non-null  str  
 2   localControlNumber_digital    44828 non-null  str  
 3   marcDisplayText_digital       44828 non-null  str  
 4   instance_physical             44828 non-null  str  
 5   recordControlNumber_physical  44828 non-null  str  
 6   localControlNumber_physical   44828 non-null  str  
 7   marcDisplayText_physical      44828 non-null  str  
dtypes: str(8)
memory usage: 2.7 MB


In [72]:
physical_to_digi_matches = digitizations.merge(
    originals,
    how="inner",
    left_on=["recordControlNumber"],
    right_on=["localControlNumber"],
    suffixes=("_digital", "_physical")
)

physical_to_digi_matches.drop_duplicates(inplace=True)

physical_to_digi_matches.info()

<class 'pandas.DataFrame'>
RangeIndex: 43513 entries, 0 to 43512
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   instance_digital              43513 non-null  str  
 1   recordControlNumber_digital   43513 non-null  str  
 2   localControlNumber_digital    43513 non-null  str  
 3   marcDisplayText_digital       43513 non-null  str  
 4   instance_physical             43513 non-null  str  
 5   recordControlNumber_physical  43513 non-null  str  
 6   localControlNumber_physical   43513 non-null  str  
 7   marcDisplayText_physical      43513 non-null  str  
dtypes: str(8)
memory usage: 2.7 MB


In [73]:
cols = physical_to_digi_matches.columns

non_mutual_matches = pd.concat(
    [physical_to_digi_matches.assign(source="physical_to_digi"),
     digi_to_physical_matches.assign(source="digi_to_physical")]
    ).drop_duplicates(subset=cols, keep=False)

non_mutual_matches.info()
non_mutual_matches.head(1)

<class 'pandas.DataFrame'>
Index: 1633 entries, 27 to 44816
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   instance_digital              1633 non-null   str  
 1   recordControlNumber_digital   1633 non-null   str  
 2   localControlNumber_digital    1633 non-null   str  
 3   marcDisplayText_digital       1633 non-null   str  
 4   instance_physical             1633 non-null   str  
 5   recordControlNumber_physical  1633 non-null   str  
 6   localControlNumber_physical   1633 non-null   str  
 7   marcDisplayText_physical      1633 non-null   str  
 8   source                        1633 non-null   str  
dtypes: str(9)
memory usage: 127.6 KB


,instance_digital,recordControlNumber_digital,localControlNumber_digital,marcDisplayText_digital,instance_physical,recordControlNumber_physical,localControlNumber_physical,marcDisplayText_physical,source
27,https://libris.kb.se/sb4572m41n6p35m#it,11716654,11716156,Originalversion:,https://libris.kb.se/p7124zj11973jnl#it,11716651,11716654,Digitaliserad version:,physical_to_digi


- Rader med source "physical_to_digi" representerar fall där en fysisk instans pekar på en digital instans, som inte pekar tillbaka.

- Rader med source "digi_to_physical" representerar fall där en digital instans pekar på en fysisk instans, som inte pekar tillbaka.

In [74]:
non_mutual_matches.value_counts("source")

source
digi_to_physical    1474
physical_to_digi     159
Name: count, dtype: int64

In [75]:
non_mutual_matches.value_counts(["instance_digital", "source"])

instance_digital                          source          
https://libris.kb.se/j2vbjjtv3r846hn#it   physical_to_digi    23
https://libris.kb.se/p71cs88154gv1hj#it   physical_to_digi     4
https://libris.kb.se/dxq1sz9q2j2hxb2#it   digi_to_physical     4
https://libris.kb.se/cwp0rx8p5nnq792#it   digi_to_physical     4
https://libris.kb.se/g0s3v1cs33dpql7#it   digi_to_physical     4
                                                              ..
https://libris.kb.se/8tk0clkd6lt1r8j9#it  digi_to_physical     1
https://libris.kb.se/rb0442srpzxpqv9t#it  digi_to_physical     1
https://libris.kb.se/sc3jr4v7q3gw4vfg#it  digi_to_physical     1
https://libris.kb.se/sc3hph98qldw5hzl#it  digi_to_physical     1
https://libris.kb.se/h208dzgtfmlqg4nr#it  digi_to_physical     1
Name: count, Length: 1551, dtype: int64

In [76]:
non_mutual_matches.to_csv("non_mutual_matches.tsv", sep="\t")